# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder)without this

/Users/mac/Documents/dev/ID2221/dic/Week 3


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
from ingestion import ingest, transform, DB_SRC, PROJECT_ROOT
from Task_3 import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/25 11:31:42 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/25 11:31:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-71606972-ad51-4a7d-9ea5-7340949831be;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# drop all tables

In [3]:
from utilities.utilities import drop_all_tables

drop_all_tables(spark)


# double-check new data

In [3]:
for config_file in Path(f"{PROJECT_ROOT}/ingestion_update_configuration").glob("*.json"):
    print(f"Processing config file: {config_file}")
    df, config = ingest(spark, config_file)
    new_df = transform(df, config)

    new_df.printSchema()
    new_df.show(truncate=False)

Processing config file: /Users/mac/Documents/dev/ID2221/dic/ingestion_update_configuration/air_quality.json


26/09/25 11:31:54 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/mac/Documents/dev/ID2221/dic/updates/air_quality_update/*.csv.
java.io.FileNotFoundException: File /Users/mac/Documents/dev/ID2221/dic/updates/air_quality_update/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$

root
 |-- county: string (nullable = true)
 |-- datetime: timestamp (nullable = false)
 |-- measurement: float (nullable = true)
 |-- aqi: float (nullable = true)

+------------+-------------------+-----------+-----+
|county      |datetime           |measurement|aqi  |
+------------+-------------------+-----------+-----+
|Jefferson   |2025-01-07 21:00:00|12.580212  |189.0|
|Cache       |2025-02-07 18:00:00|1.9884071  |441.0|
|Swain       |2025-02-08 02:00:00|2.9130187  |333.0|
|Jefferson   |2025-02-16 07:00:00|10.352773  |434.0|
|Sumter      |2025-02-17 03:00:00|3.9109704  |5.0  |
|Cache       |2025-02-20 22:00:00|19.022097  |441.0|
|Sullivan    |2025-02-22 08:00:00|6.4462867  |400.0|
|Swain       |2025-03-15 10:00:00|10.924814  |333.0|
|Jefferson   |2025-03-16 21:00:00|6.8585715  |189.0|
|Cache       |2025-03-18 16:00:00|11.167275  |441.0|
|Ozaukee     |2025-03-20 12:00:00|10.742693  |481.0|
|Mitchell    |2025-03-21 23:00:00|17.324375  |329.0|
|DeKalb      |2025-03-30 12:00:00|4.64513

26/09/25 11:31:54 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/mac/Documents/dev/ID2221/dic/updates/taxi_trips_update/*.parquet.
java.io.FileNotFoundException: File /Users/mac/Documents/dev/ID2221/dic/updates/taxi_trips_update/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataS

root
 |-- pu_datetime: timestamp (nullable = true)
 |-- do_datetime: timestamp (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- trip_distance: float (nullable = true)

+-------------------+-------------------+--------------+--------------+-----------+-------------+
|pu_datetime        |do_datetime        |pu_location_id|do_location_id|fare_amount|trip_distance|
+-------------------+-------------------+--------------+--------------+-----------+-------------+
|2024-01-01 00:20:11|2024-01-01 00:42:53|4             |238           |28.9       |5.88         |
|2024-01-01 00:15:34|2024-01-01 00:28:51|162           |143           |14.2       |2.1          |
|2024-01-01 00:36:26|2024-01-01 01:08:23|148           |244           |47.8       |11.48        |
|2024-01-01 00:44:37|2024-01-01 00:56:29|68            |107           |11.4       |1.01         |
|2024-01-01 00:24:05|2024-01-01 00:3

26/09/25 11:31:55 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/mac/Documents/dev/ID2221/dic/updates/weather_update/*.csv.
java.io.FileNotFoundException: File /Users/mac/Documents/dev/ID2221/dic/updates/weather_update/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1B

+-------------------+----------+----+-----------+----+----+---------+----+----+--------+
|datetime           |temp      |rhum|prcp       |snwd|wdir|wspd     |cldc|coco|humidity|
+-------------------+----------+----+-----------+----+----+---------+----+----+--------+
|2024-12-31 00:00:00|35.6      |45  |2.2771893  |0.0 |25  |36.985504|8   |8   |45      |
|2024-12-31 01:00:00|16.09721  |49  |0.3064257  |0.0 |136 |18.394165|4   |4   |49      |
|2024-12-31 02:00:00|21.07102  |33  |0.793313   |0.0 |260 |22.987251|2   |14  |33      |
|2024-12-31 03:00:00|9.535672  |57  |0.0        |0.0 |40  |12.334884|3   |3   |57      |
|2024-12-31 04:00:00|33.25072  |73  |1.9855865  |0.0 |160 |34.23465 |7   |7   |73      |
|2024-12-31 05:00:00|16.258282 |49  |0.322193   |0.0 |136 |18.542908|4   |4   |49      |
|2024-12-31 06:00:00|9.296201  |89  |0.0        |0.0 |315 |12.113743|0   |NULL|89      |
|2024-12-31 07:00:00|31.85386  |34  |1.8488476  |0.0 |103 |32.944714|1   |1   |34      |
|2024-12-31 08:00:00|

# Make sure schema_versions have been noted for the initial datasets

In [5]:
for config_file in Path(f"{PROJECT_ROOT}/ingestion_update_configuration").glob("*.json"):
    print(f"Processing config file: {config_file}")
    with open(config_file) as f:
        config = json.load(f)

    print(f"current schema version for {config['name']}: {register_schema_version(spark, config['name'])}")

spark.table("schema_versions").show(truncate=False)

Processing config file: /Users/mac/Documents/dev/ID2221/dic/ingestion_update_configuration/air_quality.json
Current Delta version for air_quality: 0
Current schema for air_quality: {"fields":[{"metadata":{},"name":"county","nullable":true,"type":"string"},{"metadata":{},"name":"datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"measurement","nullable":true,"type":"float"}],"type":"struct"}
current schema version for air_quality: 0
Processing config file: /Users/mac/Documents/dev/ID2221/dic/ingestion_update_configuration/taxi_trips.json
Current Delta version for taxi_trips: 0
Current schema for taxi_trips: {"fields":[{"metadata":{},"name":"pu_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"do_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"pu_location_id","nullable":true,"type":"integer"},{"metadata":{},"name":"do_location_id","nullable":true,"type":"integer"},{"metadata":{},"name":"fare_amount","nullable":true,"type":"float"},

# Ingest new data

In [10]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

for config_file in Path(f"{PROJECT_ROOT}/ingestion_update_configuration").glob("*.json"):
    print(f"Processing config file: {config_file}")
    df, config = ingest(spark, config_file)
    new_df = transform(df, config)

    new_df.printSchema()
    new_df.show(truncate=False)

    with log_step("write_delta"):
        # persist delta table
        from delta.tables import DeltaTable
        from functools import reduce

        update_pipeline_execution(spark, new_df, config['name'])



Processing config file: /Users/mac/Documents/dev/ID2221/dic/ingestion_update_configuration/air_quality.json
root
 |-- county: string (nullable = true)
 |-- datetime: timestamp (nullable = false)
 |-- measurement: float (nullable = true)
 |-- aqi: float (nullable = true)

+------------+-------------------+-----------+-----+
|county      |datetime           |measurement|aqi  |
+------------+-------------------+-----------+-----+
|Jefferson   |2025-01-07 21:00:00|12.580212  |189.0|
|Cache       |2025-02-07 18:00:00|1.9884071  |441.0|
|Swain       |2025-02-08 02:00:00|2.9130187  |333.0|
|Jefferson   |2025-02-16 07:00:00|10.352773  |434.0|
|Sumter      |2025-02-17 03:00:00|3.9109704  |5.0  |
|Cache       |2025-02-20 22:00:00|19.022097  |441.0|
|Sullivan    |2025-02-22 08:00:00|6.4462867  |400.0|
|Swain       |2025-03-15 10:00:00|10.924814  |333.0|
|Jefferson   |2025-03-16 21:00:00|6.8585715  |189.0|
|Cache       |2025-03-18 16:00:00|11.167275  |441.0|
|Ozaukee     |2025-03-20 12:00:00|10.74

26/09/25 11:34:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/mac/Documents/dev/ID2221/dic/updates/air_quality_update/*.csv.
java.io.FileNotFoundException: File /Users/mac/Documents/dev/ID2221/dic/updates/air_quality_update/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$

Processing config file: /Users/mac/Documents/dev/ID2221/dic/ingestion_update_configuration/taxi_trips.json
root
 |-- pu_datetime: timestamp (nullable = true)
 |-- do_datetime: timestamp (nullable = true)
 |-- pu_location_id: integer (nullable = true)
 |-- do_location_id: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- trip_distance: float (nullable = true)

+-------------------+-------------------+--------------+--------------+-----------+-------------+
|pu_datetime        |do_datetime        |pu_location_id|do_location_id|fare_amount|trip_distance|
+-------------------+-------------------+--------------+--------------+-----------+-------------+
|2024-01-01 00:20:11|2024-01-01 00:42:53|4             |238           |28.9       |5.88         |
|2024-01-01 00:15:34|2024-01-01 00:28:51|162           |143           |14.2       |2.1          |
|2024-01-01 00:36:26|2024-01-01 01:08:23|148           |244           |47.8       |11.48        |
|2024-01-01 00:44:37|2024-0

26/09/25 11:35:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/mac/Documents/dev/ID2221/dic/updates/taxi_trips_update/*.parquet.
java.io.FileNotFoundException: File /Users/mac/Documents/dev/ID2221/dic/updates/taxi_trips_update/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataS

Processing config file: /Users/mac/Documents/dev/ID2221/dic/ingestion_update_configuration/weather.json
root
 |-- datetime: timestamp (nullable = false)
 |-- temp: float (nullable = true)
 |-- rhum: integer (nullable = true)
 |-- prcp: float (nullable = true)
 |-- snwd: float (nullable = true)
 |-- wdir: integer (nullable = true)
 |-- wspd: float (nullable = true)
 |-- cldc: integer (nullable = true)
 |-- coco: integer (nullable = true)
 |-- humidity: integer (nullable = true)

+-------------------+----------+----+-----------+----+----+---------+----+----+--------+
|datetime           |temp      |rhum|prcp       |snwd|wdir|wspd     |cldc|coco|humidity|
+-------------------+----------+----+-----------+----+----+---------+----+----+--------+
|2024-12-31 00:00:00|35.6      |45  |2.2771893  |0.0 |25  |36.985504|8   |8   |45      |
|2024-12-31 01:00:00|16.09721  |49  |0.3064257  |0.0 |136 |18.394165|4   |4   |49      |
|2024-12-31 02:00:00|21.07102  |33  |0.793313   |0.0 |260 |22.987251|2  

26/09/25 11:35:07 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/mac/Documents/dev/ID2221/dic/updates/weather_update/*.csv.
java.io.FileNotFoundException: File /Users/mac/Documents/dev/ID2221/dic/updates/weather_update/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1B

# Check if schema_versions have changed since update

In [11]:
spark.table("schema_versions").show(truncate=False)

+-----------+--------------+-------------+----------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+
|table_name |schema_version|delta_version|schema_hash                                                     |schema_json                                                                                      

# Check pipeline monitor table

In [12]:
read_pipeline_monitor(spark).show(truncate=False)

+--------------------------+--------------------------+-----------------+----------------+----------------+-------------------+-----------+--------------+-------------+
|execution_start_time      |execution_end_time        |processed_records|inserted_records|rejected_records|validation_failures|table_name |schema_version|delta_version|
+--------------------------+--------------------------+-----------------+----------------+----------------+-------------------+-----------+--------------+-------------+
|2026-09-25 11:33:50.214113|2026-09-25 11:33:59.685581|813955           |813955          |0               |NULL               |air_quality|1             |1            |
|2026-09-25 11:34:03.617488|2026-09-25 11:34:08.61638 |355937           |296462          |59475           |NULL               |taxi_trips |0             |1            |
|2026-09-25 11:34:10.89671 |2026-09-25 11:34:12.379842|8784             |8784            |0               |NULL               |weather    |1             |1

# Table histories
Note: The merge only occurs once since subsequent attempts on the same update data is recognized as duplicates

In [11]:
# see deltalog history
spark.sql("DESCRIBE HISTORY default.air_quality").show(truncate=False)


+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
# see deltalog history
spark.sql("DESCRIBE HISTORY default.taxi_trips").show(truncate=False)


+-------+-----------------------+------+--------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
# see deltalog history
spark.sql("DESCRIBE HISTORY default.weather").show(truncate=False)


+-------+-----------------------+------+--------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Table content

In [ ]:
# show the delta table content
spark.table("default.air_quality").show(truncate=False)

In [ ]:
# show the delta table content
spark.table("default.taxi_trips").show(truncate=False)

In [ ]:
# show the delta table content
spark.table("default.weather").show(truncate=False)